## 1. Download Dataset from Kaggle

The retail orders dataset is downloaded from Kaggle using the Kaggle CLI.
The dataset is then extracted locally for further processing using Pandas.

In [ ]:
# Download the dataset from Kaggle
!kaggle datasets download ankitbansal06/retail-orders -f orders.csv

In [ ]:
# Extract the downloaded ZIP file
import zipfile

with zipfile.ZipFile("orders.csv.zip", "r") as zip_ref:
    zip_ref.extractall()

In [ ]:
import pandas as pd

df = pd.read_csv('orders.csv', na_values=['Not Available','unknown']) ## Treat specified placeholder values as missing values
df['Ship Mode'].unique()

In [2]:
# Standardize column names
df.columns = df.columns.str.lower()
df.columns = df.columns.str.replace(" ", "_", regex=False)

In [4]:
# Derive discount, sale price and profit
df["discount"] = df["list_price"] * df["discount_percent"] * 0.01
df["sale_price"] = df["list_price"] - df["discount"]
df["profit"] = df["sale_price"] - df["cost_price"]

In [5]:
#convert order date from object data type to datetime
df['order_date'] = pd.to_datetime(df['order_date'], format='%Y-%m-%d')


In [7]:
df[["discount", "sale_price", "profit"]].head()

,discount,sale_price,profit
0,5.2,254.8,14.8
1,21.9,708.1,108.1
2,0.5,9.5,-0.5
3,19.2,940.8,160.8
4,1.0,19.0,-1.0


In [ ]:
# Remove source pricing columns after deriving calculated metrics
df.drop(columns=["cost_price", "list_price", "discount_percent"],inplace=True)

,order_id,order_date,ship_mode,segment,country,city,state,postal_code,region,category,sub_category,product_id,quantity,discount,sale_price,profit
0,1,2023-03-01,Second Class,Consumer,United States,Henderson,Kentucky,42420,South,Furniture,Bookcases,FUR-BO-10001798,2,5.2,254.8,14.8
1,2,2023-08-15,Second Class,Consumer,United States,Henderson,Kentucky,42420,South,Furniture,Chairs,FUR-CH-10000454,3,21.9,708.1,108.1
2,3,2023-01-10,Second Class,Corporate,United States,Los Angeles,California,90036,West,Office Supplies,Labels,OFF-LA-10000240,2,0.5,9.5,-0.5
3,4,2022-06-18,Standard Class,Consumer,United States,Fort Lauderdale,Florida,33311,South,Furniture,Tables,FUR-TA-10000577,5,19.2,940.8,160.8
4,5,2022-07-13,Standard Class,Consumer,United States,Fort Lauderdale,Florida,33311,South,Office Supplies,Storage,OFF-ST-10000760,2,1.0,19.0,-1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9989,9990,2023-02-18,Second Class,Consumer,United States,Miami,Florida,33180,South,Furniture,Furnishings,FUR-FU-10001889,3,1.2,28.8,-1.2
9990,9991,2023-03-17,Standard Class,Consumer,United States,Costa Mesa,California,92627,West,Furniture,Furnishings,FUR-FU-10000747,2,3.6,86.4,16.4
9991,9992,2022-08-07,Standard Class,Consumer,United States,Costa Mesa,California,92627,West,Technology,Phones,TEC-PH-10003645,2,5.2,254.8,34.8
9992,9993,2022-11-19,Standard Class,Consumer,United States,Costa Mesa,California,92627,West,Office Supplies,Paper,OFF-PA-10004041,4,0.9,29.1,-0.9


## 2. Connect to SQL Server

Connect to the local SQL Server database using SQLAlchemy, pyodbc, and Windows Authentication.

In [8]:
import sqlalchemy as sal
import pyodbc

engine = sal.create_engine(
    "mssql+pyodbc://localhost/PracticeDB"
    "?driver=ODBC+Driver+18+for+SQL+Server"
    "&Trusted_Connection=yes"
    "&TrustServerCertificate=yes"
)

conn = engine.connect()

print("Connected successfully!")

Connected successfully!


## 3. Load Data into SQL Server

Load the transformed Pandas DataFrame into the `df_orders` table in the `PracticeDB` database.

In [ ]:
df.to_sql('df_orders', con=conn, index=False, if_exists = 'append')